In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

In [2]:
print('Loading data...')
train = pd.read_csv('../data/raw_data/train.csv')
test = pd.read_csv('../data/raw_data/test.csv')

train['Will_Buy_EV'] = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X = train.drop(columns=['Will_Buy_EV', 'id'], errors='ignore')
y = train['Will_Buy_EV']
X_test = test.drop(columns=['id'], errors='ignore')
test_ids = test['id']

Loading data...


In [3]:
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

C:\Users\Vajresh\AppData\Local\Temp\ipykernel_12168\4022144355.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cat_test_preds = np.zeros(len(X_test))
cv_scores = []

In [5]:
print('Training CatBoost...')
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500, learning_rate=0.05, cat_features=cat_features,
        eval_metric='AUC', random_seed=42, verbose=False
    )

    # CatBoost can evaluate early stopping natively
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50)
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_preds)
    cv_scores.append(auc)
    print(f'Fold {fold+1} AUC: {auc:.5f}')

    cat_test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

print(f'\nMean AUC: {np.mean(cv_scores):.5f}')

Training CatBoost...
Fold 1 AUC: 0.94003
Fold 2 AUC: 0.94102
Fold 3 AUC: 0.94222
Fold 4 AUC: 0.94190
Fold 5 AUC: 0.94125

Mean AUC: 0.94128


In [6]:
submission = pd.DataFrame({'id': test_ids, 'Will_Buy_EV': cat_test_preds})
os.makedirs('../output', exist_ok=True)
sub_path = '../output/submission_catboost.csv'
submission.to_csv(sub_path, index=False)
print(f'Saved to {sub_path}')

Saved to ../output/submission_catboost.csv
